# Trump Tweet Market Impact Classifier — Training Notebook

## Multi-Layer Inference System

**Architecture:**
```
┌──────────────────────────────────────────────────────────┐
│  Layer 1: Entity Extraction (GLiNER zero-shot NER)       │
│  → people, countries, commodities, currencies, orgs      │
├──────────────────────────────────────────────────────────┤
│  Layer 2: Financial Sentiment (FinBERT)                  │
│  → positive/negative/neutral + compound score            │
├──────────────────────────────────────────────────────────┤
│  Layer 3: Event Detection (DistilBERT + rules)           │
│  → trade_war, sanctions, monetary_policy, ...            │
├──────────────────────────────────────────────────────────┤
│  Layer 4: Context Enrichment (temporal + Tavily)         │
│  → market hours, tweet patterns, macro context           │
├──────────────────────────────────────────────────────────┤
│  Layer 5: Graph Reasoning (entity-event-asset graphs)    │
│  → transmission rules, signal propagation                │
├──────────────────────────────────────────────────────────┤
│  Layer 6: Market Impact (LightGBM per asset/timeframe)   │
│  → direction {-1,0,1} + confidence [0,1]                 │
└──────────────────────────────────────────────────────────┘
```

**Assets:** Gold (GC) · Equities (ES) · BTC · Crude Oil (CL) · Wheat (ZW) · EuroDollar (6E) · Treasury 2Y (ZT)  
**Timeframes:** 1m, 5m, 10m post-tweet

## 1. Setup & Dependencies

In [ ]:
import subprocess, sys

deps = [
    "torch", "transformers", "gliner", "lightgbm", "scikit-learn",
    "pandas", "numpy", "matplotlib", "seaborn", "networkx", "tqdm",
    "shap",
]

for pkg in deps:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All dependencies ready.")

In [ ]:
import os
import sys
import json
import time
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('training')

# Ensure project modules are importable
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load & Explore Training Data

In [ ]:
DATA_PATH = "data/train.csv"
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# Asset-specific label columns
ASSETS = ["gold", "equities", "btc", "cl", "wheat", "eurodollar", "treasury_2y"]
TICKERS = {"gold": "GC", "equities": "ES", "btc": "BTC", "cl": "CL",
           "wheat": "ZW", "eurodollar": "6E", "treasury_2y": "ZT"}
TIMEFRAMES = ["1m", "5m", "10m"]

# Check label distributions
print("LABEL DISTRIBUTIONS (actual directions):")
print(f"{'Asset':<14} {'TF':<4} {'Bear(-1)':>10} {'Flat(0)':>10} {'Bull(1)':>10}")
print("-" * 55)
for asset in ASSETS:
    for tf in TIMEFRAMES:
        col = f"{asset}_{TICKERS[asset]}_actual_dir_{tf}"
        vc = df[col].value_counts()
        print(f"{asset:<14} {tf:<4} {vc.get(-1, 0):>10} {vc.get(0, 0):>10} {vc.get(1, 0):>10}")

In [ ]:
# Market relevance
print("Market Relevance Distribution:")
print(df["is_market_relevant"].value_counts())
print(f"\nRelevance score stats:")
print(df["relevance_score"].describe())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Relevance score dist
df["relevance_score"].hist(bins=30, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Relevance Score Distribution")
axes[0].set_xlabel("Relevance Score")

# Label balance for equities 5m
col = "equities_ES_actual_dir_5m"
df[col].value_counts().sort_index().plot(kind="bar", ax=axes[1], color=["#e74c3c","#95a5a6","#2ecc71"])
axes[1].set_title("Equities 5m Direction")
axes[1].set_xticklabels(["-1 (Bear)", "0 (Flat)", "1 (Bull)"], rotation=0)

# Tweet length distribution
df["content"].str.len().hist(bins=50, ax=axes[2], color="orange", edgecolor="white")
axes[2].set_title("Tweet Length Distribution")
axes[2].set_xlabel("Characters")

plt.tight_layout()
plt.show()

## 3. Layer 1 — Entity Extraction (GLiNER)

In [ ]:
from models.ner.entity_extractor import EntityExtractor

ner = EntityExtractor(use_gpu=torch.cuda.is_available())
ner.load()

# Test on sample tweets
test_tweets = [
    "I have just ordered 50% TARIFFS on all goods from China, effective immediately!",
    "Great job by the Republican Party! We are winning BIG!",
    "Iran situation is getting worse. Oil prices too high. Told OPEC to pump more!",
    "Bitcoin is very exciting. Working on crypto regulation framework.",
]

for tweet in test_tweets:
    result = ner.extract(tweet)
    print(f"\n'{tweet[:80]}...'")
    for ent in result.entities:
        print(f"  {ent.label:20s} | {ent.text:20s} | score={ent.score:.2f} | assets={ent.mapped_assets}")
    print(f"  Asset relevance: {result.asset_relevance}")

## 4. Layer 2 — Financial Sentiment (FinBERT)

In [ ]:
from models.sentiment.finbert_sentiment import FinBERTSentiment

sentiment = FinBERTSentiment(use_gpu=torch.cuda.is_available())
sentiment.load()

for tweet in test_tweets:
    result = sentiment.analyze(tweet)
    print(f"\n'{tweet[:70]}...'")
    print(f"  Label: {result.label:10s} | Score: {result.score:.3f} | "
          f"Compound: {result.compound:+.3f}")
    print(f"  Pos: {result.positive:.3f} | Neg: {result.negative:.3f} | Neu: {result.neutral:.3f}")

## 5. Layer 3 — Event Detection

In [ ]:
from models.events.event_detector import EventDetector

event_det = EventDetector(use_gpu=torch.cuda.is_available())
event_det.load()

for tweet in test_tweets:
    result = event_det.detect_rules(tweet)
    print(f"\n'{tweet[:70]}...'")
    print(f"  Primary: {result.primary_event} (conf={result.primary_confidence:.2f})")
    print(f"  All events: {result.events}")

## 6. Full Feature Extraction Pipeline

In [ ]:
from models.events.context_enrichment import ContextEnricher
from models.graph_reasoning.entity_graph import EntityGraph

context_enricher = ContextEnricher()
graph_reasoner = EntityGraph()

texts = df["content"].fillna("").astype(str).tolist()
timestamps = df["created_at"].fillna("").astype(str).tolist()

print(f"Extracting features for {len(texts)} tweets...")
t0 = time.time()

all_features = []
for i in tqdm(range(len(texts)), desc="Feature extraction"):
    text = texts[i]
    ts = timestamps[i]
    prev_ts = timestamps[i - 1] if i > 0 else None
    
    features = {}
    
    # Layer 1: NER
    ner_result = ner.extract(text)
    features.update(ner_result.to_feature_dict())
    
    # Layer 2: Sentiment
    sent_result = sentiment.analyze(text)
    features.update(sent_result.to_feature_dict())
    
    # Layer 3: Events (rules first, trained head later)
    event_result = event_det.detect_rules(text)
    features.update(event_result.to_feature_dict())
    
    # Layer 4: Context
    ctx_result = context_enricher.enrich(text=text, created_at=ts, prev_tweet_time=prev_ts)
    features.update(ctx_result.to_feature_dict())
    
    # Layer 5: Graph
    try:
        graph_result = graph_reasoner.build_and_reason(
            entities=ner_result.entities,
            events=event_result.events,
            sentiment_compound=sent_result.compound,
        )
        features.update(graph_result.to_feature_dict())
    except Exception as e:
        pass
    
    all_features.append(features)

feature_time = time.time() - t0
print(f"\nDone in {feature_time:.1f}s ({feature_time/len(texts)*1000:.0f}ms/tweet)")

# Convert to DataFrame
X_df = pd.DataFrame(all_features).fillna(0.0)
print(f"Feature matrix shape: {X_df.shape}")
print(f"Features: {list(X_df.columns)}")

In [ ]:
# Save features to disk (avoids re-extraction)
X_df.to_csv("data/extracted_features.csv", index=False)
print(f"Features saved: {X_df.shape}")

# Feature statistics
print("\nFeature Statistics:")
print(X_df.describe().T[["mean", "std", "min", "max"]].to_string())

## 7. Train Event Detection Head

In [ ]:
import re
from models.events.event_detector import EVENT_TYPES, EVENT_PATTERNS

# Generate pseudo-labels from rules
event_labels = np.zeros((len(texts), len(EVENT_TYPES)), dtype=np.float32)
for i, text in enumerate(texts):
    text_lower = text.lower()
    any_event = False
    for j, event_type in enumerate(EVENT_TYPES):
        if event_type == "no_event":
            continue
        for pattern in EVENT_PATTERNS.get(event_type, []):
            if re.search(pattern, text_lower):
                event_labels[i, j] = 1.0
                any_event = True
                break
    if not any_event:
        event_labels[i, EVENT_TYPES.index("no_event")] = 1.0

# Distribution
print("Event Label Distribution:")
for j, evt in enumerate(EVENT_TYPES):
    print(f"  {evt:<25} {int(event_labels[:, j].sum()):>5} ({event_labels[:, j].mean()*100:.1f}%)")

# Train head
print("\nTraining event detection head...")
event_det.train_head(texts, event_labels, epochs=15, lr=2e-4)
event_det.save("saved_models/event_head.pt")
print("Event head trained and saved.")

In [ ]:
# Re-extract event features with trained head
print("Re-extracting event features with trained head...")
for i in tqdm(range(len(texts)), desc="Re-extract events"):
    event_result = event_det.detect(texts[i])
    event_feats = event_result.to_feature_dict()
    all_features[i].update(event_feats)

X_df = pd.DataFrame(all_features).fillna(0.0)
print(f"Updated feature matrix: {X_df.shape}")

## 8. Train Market Impact Classifiers (LightGBM)

In [ ]:
from models.market_impact.impact_predictor import MarketImpactPredictor

predictor = MarketImpactPredictor(target_timeframe="5m")

metrics = predictor.train(
    feature_dicts=all_features,
    labels_df=df,
    timeframes=["1m", "5m", "10m"],
    n_splits=5,
)

# Save
predictor.save("saved_models/impact_predictor")
print("\nModels saved to saved_models/impact_predictor/")

In [ ]:
# Results table
print("=" * 80)
print("TRAINING RESULTS")
print("=" * 80)

if "relevance" in metrics:
    m = metrics["relevance"]
    print(f"\nRelevance Classifier: Acc={m['accuracy']:.4f}, F1={m['f1']:.4f}")

rows = []
print(f"\n{'Asset':<14} {'TF':<4} {'CV Acc':>8} {'CV F1':>8} {'±Std':>8} {'Train Acc':>10}")
print("-" * 55)
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        if key in metrics:
            m = metrics[key]
            print(f"{asset:<14} {tf:<4} {m['cv_accuracy']:>8.4f} {m['cv_f1']:>8.4f} "
                  f"{m['cv_accuracy_std']:>8.4f} {m['train_accuracy']:>10.4f}")
            rows.append({"asset": asset, "tf": tf, "cv_acc": m["cv_accuracy"],
                        "cv_f1": m["cv_f1"], "train_acc": m["train_accuracy"]})

results_df = pd.DataFrame(rows)

## 9. Evaluation & Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# CV Accuracy by asset
for tf in TIMEFRAMES:
    subset = results_df[results_df["tf"] == tf]
    axes[0].barh(
        [f"{r['asset']}" for _, r in subset.iterrows()],
        subset["cv_acc"],
        alpha=0.7,
        label=tf,
    )
axes[0].set_title("CV Accuracy by Asset / Timeframe")
axes[0].set_xlim(0.3, 0.6)
axes[0].axvline(x=0.333, color="red", linestyle="--", alpha=0.5, label="Random baseline")
axes[0].legend()

# CV F1 by asset (5m)
subset_5m = results_df[results_df["tf"] == "5m"]
colors = sns.color_palette("viridis", len(ASSETS))
axes[1].barh(subset_5m["asset"], subset_5m["cv_f1"], color=colors)
axes[1].set_title("CV F1 Score (5m timeframe)")
axes[1].set_xlim(0.3, 0.55)
axes[1].axvline(x=0.333, color="red", linestyle="--", alpha=0.5)

# Overfitting check: train vs CV
subset_5m_plot = results_df[results_df["tf"] == "5m"].copy()
x_idx = np.arange(len(subset_5m_plot))
axes[2].bar(x_idx - 0.15, subset_5m_plot["cv_acc"], 0.3, label="CV Acc", color="steelblue")
axes[2].bar(x_idx + 0.15, subset_5m_plot["train_acc"], 0.3, label="Train Acc", color="coral")
axes[2].set_xticks(x_idx)
axes[2].set_xticklabels(subset_5m_plot["asset"], rotation=45)
axes[2].set_title("Train vs CV Accuracy (5m)")
axes[2].legend()

plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Feature importance analysis
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()

for idx, asset in enumerate(ASSETS):
    top_feats = predictor.get_top_features(asset, "5m", top_k=15)
    if top_feats:
        names = [f[0][:30] for f in top_feats]
        values = [f[1] for f in top_feats]
        axes[idx].barh(names[::-1], values[::-1], color=sns.color_palette("viridis", len(names)))
        axes[idx].set_title(f"{asset.upper()} — Top Features")

# Hide empty subplot
if len(ASSETS) < len(axes):
    axes[-1].set_visible(False)

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Test Inference

In [ ]:
from inference_pipeline import InferencePipeline

# Build pipeline and attach already-loaded components
pipeline = InferencePipeline(use_gpu=torch.cuda.is_available())
pipeline.entity_extractor = ner
pipeline.sentiment_analyzer = sentiment
pipeline.event_detector = event_det
pipeline.context_enricher = context_enricher
pipeline.graph_reasoner = graph_reasoner
pipeline.impact_predictor = predictor
pipeline._loaded = True

# Test predictions
test_cases = [
    ("I have just ordered 50% TARIFFS on all goods from China, effective immediately!", "2025-04-09T14:30:00Z"),
    ("Great meeting with President Xi. Trade deal looking very good!", "2025-06-15T10:00:00Z"),
    ("Oil prices are too high. Iran situation escalating.", "2025-03-20T08:00:00Z"),
    ("Bitcoin is very exciting. Working on crypto regulation framework.", "2025-05-01T12:00:00Z"),
    ("Great job by the Republican Party! We are winning BIG!", "2025-07-04T16:00:00Z"),
]

for tweet, ts in test_cases:
    result = pipeline.predict(tweet, created_at=ts)
    print(f"\n{'='*70}")
    print(f"Tweet: \"{tweet[:80]}...\"")
    print(f"Relevant: {result.is_market_relevant} (score={result.relevance_score:.2f})")
    print(f"\n  {'Asset':<14} {'Dir':>5} {'Conf':>6}  Reasoning")
    print(f"  {'-'*60}")
    for asset in ASSETS:
        p = result.predictions[asset]
        arrow = {1: '↑', -1: '↓', 0: '→'}[p.direction]
        print(f"  {asset:<14} {arrow:>5} {p.confidence:>6.3f}  {'; '.join(p.reasoning[:2])}")

## 11. SHAP Explainability

In [ ]:
try:
    import shap
    
    # For equities 5m model
    model_eq5 = predictor.models.get(("equities", "5m"))
    if model_eq5:
        X_shap = X_df.head(500)  # Use subset for speed
        explainer = shap.TreeExplainer(model_eq5)
        shap_values = explainer.shap_values(X_shap)
        
        plt.figure(figsize=(12, 8))
        # shap_values is list of 3 arrays (one per class)
        # Class 2 = bullish (mapped from 1)
        shap.summary_plot(shap_values[2], X_shap, max_display=20, show=False)
        plt.title("SHAP Values — Equities 5m (Bullish Class)")
        plt.tight_layout()
        plt.savefig("shap_equities_5m.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("No equities/5m model found")
except ImportError:
    print("Install shap for explainability: pip install shap")
except Exception as e:
    print(f"SHAP analysis failed: {e}")

## 12. Save Everything

In [ ]:
# Save metrics
with open("saved_models/training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)

print("Saved:")
print("  - saved_models/impact_predictor/  (LightGBM models)")
print("  - saved_models/event_head.pt      (Event detection head)")
print("  - saved_models/training_metrics.json")
print("  - data/extracted_features.csv")
print("  - training_results.png")
print("  - feature_importance.png")

print(f"\nTotal models: {len(predictor.models)} direction classifiers + 1 relevance")
print(f"Feature count: {len(predictor.feature_names)}")
print("\nTo use for inference:")
print("  pipeline = InferencePipeline.load('saved_models/')")
print("  result = pipeline.predict('TARIFFS on China!')")